# 02. Параметризация таргета и базовые модели

По результатам EDA релаксационные члены представляются в виде

$$R = A(T, T_{12}, T_3)\,\bigl[1 - e^{-\varphi}\bigr]$$

где сродство $\varphi$ вычисляется по спектроскопическим константам, а $A$ —
гладкий множитель, подлежащий аппроксимации.

Задачи:
1. извлечь $A$ из данных и проверить, что она одного знака и гладкая;
2. измерить, что даёт факторизация по сравнению с прямой аппроксимацией $R$;
3. получить базовые метрики, с которыми будет сравниваться нейросеть.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from co2relax.data import load_core, TARGETS, FIG_DIR
from co2relax.physics import AFFINITY, factor, THETA_2, THETA_3, DELTA_VV23, DELTA_VV123
from co2relax.splits import SPLITS, split_subsample
from co2relax.baselines import extract_A, GridSpline, ScatteredSpline, run_baseline, metrics

pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4g}".format)
FIG_DIR.mkdir(parents=True, exist_ok=True)

core = load_core()
print(core.shape)
core.head()

## 1. Извлечение множителя A

$A = R / [1 - e^{-\varphi}]$. На равновесии обе части обращаются в ноль,
поэтому там $A$ не определён — эти точки дают NaN.

Проверяем три вещи: знак постоянен, разброс умеренный, выбросов у нулевой
поверхности нет (там знаменатель мал, и деление могло бы взорваться).

In [ ]:
rows = []
for t in TARGETS:
    A = extract_A(core, t)
    a = A.dropna()
    s = np.sign(a)
    rows.append(dict(
        канал=t,
        определён=len(a),
        NaN=int(A.isna().sum()),
        знак_постоянен=bool((s == s.iloc[0]).all()),
        мин=a.abs().min(),
        макс=a.abs().max(),
        порядков=float(np.log10(a.abs().max() / a.abs().min())),
    ))
pd.DataFrame(rows)

In [ ]:
for t in TARGETS:
    print(t, extract_A(core, t).dropna().iloc[0])

Если знак постоянен — параметризация верна: весь знак и вся нулевая поверхность
сидят в множителе $[1 - e^{-\varphi}]$, который считается аналитически.

Если у какого-то канала знак сплошь противоположный остальным — это лишь
соглашение о направлении $\varphi$; достаточно поменять знак в соответствующей
функции сродства. Плохо только если знак **меняется внутри** канала.

In [ ]:
# Где A не определён — это в точности равновесные точки
A = extract_A(core, "R_VV23_12")
core.loc[A.isna(), ["T", "T12", "T3"]]

## 2. Насколько A гладкая

Сравниваем с исходным $R$: у того две плато и резкий переход, у $A$ ожидается
монотонная поверхность без особенностей.

In [ ]:
T_fix = sorted(core["T"].unique())[0]
sub = core[core["T"] == T_fix]
Acol = extract_A(core, "R_VV23_12")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
R0 = np.median(np.abs(core["R_VV23_12"][core["R_VV23_12"] != 0]))

for t12, g in sub.groupby("T12"):
    g = g.sort_values("T3")
    axes[0].plot(g["T3"], np.arcsinh(g["R_VV23_12"] / R0), "o-", ms=3)
    axes[1].plot(g["T3"], Acol.loc[g.index].abs(), "o-", ms=3, label=f"{t12:g}")

axes[0].axhline(0, color="k", lw=0.6)
axes[0].set_title(r"исходный $R_{VV23,12}$")
axes[0].set_ylabel(r"$\mathrm{asinh}(R/R_0)$")
axes[1].set_yscale("log")
axes[1].set_title(r"множитель $|A|$")
for ax in axes:
    ax.set_xlabel(r"$T_3$, К")
    ax.grid(alpha=0.3)
axes[1].legend(title=r"$T_{12}$, К", fontsize=7, ncol=2)
fig.suptitle(f"T = {T_fix:g} К")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_A_vs_R.png", dpi=150, bbox_inches="tight")

In [ ]:
d = core.assign(phi=AFFINITY["R_VV23_12"](core["T"], core["T12"], core["T3"]),
                A=extract_A(core, "R_VV23_12")).sort_values(["T","T12","T3"])

g = d.groupby(["T","T12"])["A"]
d["nb"] = (g.shift(1) + g.shift(-1)) / 2      # среднее двух соседей по T3
d["откл"] = (d["A"] / d["nb"] - 1).abs()
sub = d.dropna(subset=["откл","A"])

print(sub.nlargest(8, "откл")[["T","T12","T3","phi","откл"]])
print()
for lo, hi in [(0,.05), (.05,.2), (.2,.5), (.5,9)]:
    m = sub[(sub["phi"].abs() >= lo) & (sub["phi"].abs() < hi)]
    print(f"|phi| в [{lo:.2f},{hi:.2f}): n ={len(m):4d}  медиана {m['откл'].median():.2e}")

In [ ]:
# Карта A в плоскости (T12, T3)
fig, axs = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, t in zip(axs, TARGETS):
    d = core.assign(A=extract_A(core, t).abs())
    piv = d[d["T"] == T_fix].pivot_table(index="T12", columns="T3", values="A")
    im = ax.pcolormesh(piv.columns, piv.index, np.log10(piv.values), shading="nearest")
    fig.colorbar(im, ax=ax, label=r"$\log_{10}|A|$")
    ax.set_title(t); ax.set_xlabel(r"$T_3$, К")
axs[0].set_ylabel(r"$T_{12}$, К")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_A_maps.png", dpi=150, bbox_inches="tight")

## 3. Базовые модели

Сплайн-интерполяция на сетке — честный и сильный соперник для нейросети:
данные лежат на регулярной решётке, функция гладкая.

Два варианта:
- **по R** — интерполируется сам релаксационный член;
- **по A** — интерполируется множитель, затем $R$ восстанавливается
  умножением на $[1 - e^{-\varphi}]$.

Разница между ними и есть вклад физической параметризации.

### Разбиения

Случайный split на регулярной сетке даёт утечку: у каждой тестовой точки
соседи в 500 К остаются в обучении. Поэтому разбиения структурные:

| Разбиение | Что проверяет |
|---|---|
| прореживание сетки | интерполяция при вдвое более крупном шаге |
| срезы по T | обобщение на невиденные значения T |
| срезы по T12 | то же по T12 |
| экстраполяция по T3 | поведение за границей обучающей области |

In [ ]:
rows = []
for sname, fn in SPLITS.items():
    tr, te = fn(core)
    for t in TARGETS:
        for mode, on_A in [("по R", False), ("по A", True)]:
            try:
                m = run_baseline(tr, te, t, model=GridSpline(), on_A=on_A)
                used = "grid-cubic"
            except ValueError:          # решётка неполная — переходим на scattered
                m = run_baseline(tr, te, t, model=ScatteredSpline(), on_A=on_A)
                used = "scattered"
            rows.append(dict(разбиение=sname, канал=t, режим=mode, модель=used,
                             n_test=m["n"], rel_med=m["rel_median"],
                             rel_p95=m["rel_p95"], r2=m["r2"]))

res = pd.DataFrame(rows)
res.to_csv(FIG_DIR.parent / "02_baselines.csv", index=False)
res.pivot_table(index=["разбиение", "канал"], columns="режим",
                values=["rel_med", "rel_p95"])

In [ ]:
# Во сколько раз факторизация уменьшает ошибку
p = res.pivot_table(index=["разбиение", "канал"], columns="режим", values="rel_med")
(p["по R"] / p["по A"]).round(1).rename("выигрыш, раз").to_frame()

In [ ]:
rows = []
for sname, fn in SPLITS.items():
    tr, te = fn(core)
    eq = te[(te["T"] == te["T12"]) & (te["T12"] == te["T3"])]
    if not len(eq):
        continue
    for t in TARGETS:
        scale = np.median(np.abs(core[t][core[t] != 0]))
        for mode, on_A in [("по R", False), ("по A", True)]:
            m = GridSpline()
            col = t
            if on_A:
                tr2 = tr.assign(_A=extract_A(tr, t)); col = "_A"
            else:
                tr2 = tr
            m.fit(tr2, col)
            p = m.predict(eq)
            if on_A:
                p = p * factor(AFFINITY[t](eq["T"], eq["T12"], eq["T3"]))
            rows.append(dict(разбиение=sname, канал=t, режим=mode,
                             невязка=np.abs(p).max() / scale))
pd.DataFrame(rows).pivot_table(index=["разбиение","канал"], columns="режим", values="невязка")

### Точность аппроксимации

Прямая сплайн-интерполяция релаксационных членов превосходит интерполяцию предэкспоненциального множителя: для канала VT2 — в 20–440 раз, для VV-каналов результаты сопоставимы. Причина в обусловленности извлечения

$$
A = \frac{R}{1-e^{-\varphi}}.
$$

Для VT2 нулевая поверхность представляет собой плоскость $T_{12}=T$ (11\% узлов сетки), вблизи которой множитель убывает до $0.02$, а глобальный характер кубической интерполяции распространяет возникающее искажение на всю область.

### Физические ограничения

По невязке на равновесии соотношение обратное: факторизация обеспечивает машинный ноль ($10^{-16}$), прямая интерполяция даёт до $2\cdot10^{-2}$ от масштаба $R$. Оценка показывает, что источник такой величины изменяет заселённость моды за время порядка $10\,\mu\text{с}$ при газодинамических временах порядка $1\,\text{мс}$, то есть делает расчёт равновесных режимов непригодным.

### Следствие для архитектуры

Множитель

$$
1-e^{-\varphi}
$$

включается в модель структурно:

$$
R_{\mathrm{pred}} = NN(x)\cdot\left(1-e^{-\varphi}\right),
$$

а функция потерь вычисляется по $R$.

Такая постановка исключает деление на малую величину, сохраняя точное выполнение ограничений. Сплайн-интерполяция принципиально не допускает встраивания подобной структуры, что и определяет область применимости нейросетевого подхода.